# 01 — Data Preprocessing

**Purpose:** Load and clean both Tobii Pro Lab TSV exports into analysis-ready Parquet files.

**Inputs:**
- `data/raw/VisualTask with recording Metrics.tsv`
- `data/raw/VisualTask with recording Data export.tsv`

**Outputs:**
- `data/processed/metrics_clean.parquet`
- `data/processed/raw_gaze_clean.parquet`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path

from src.config import (
    METRICS_TSV, DATAEXPORT_TSV,
    METRICS_CLEAN_PKL, RAWGAZE_CLEAN_PKL,
    MetricsCols, ExportCols, VALID_EYE_MOVEMENT_TYPES, TASKS
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)

## 1. Load Metrics TSV

In [ ]:
metrics_raw = pd.read_csv(METRICS_TSV, sep='\t', low_memory=False)
print(f"Shape: {metrics_raw.shape}")
print(f"Columns ({len(metrics_raw.columns)}):")
metrics_raw.columns.tolist()

In [ ]:
# Null audit
null_pct = metrics_raw.isnull().mean().round(3) * 100
print("Null % per column:")
null_pct[null_pct > 0].sort_values(ascending=False)

## 2. Clean Metrics TSV

In [ ]:
metrics = metrics_raw.copy()

# Standardize participant ID
metrics[MetricsCols.PARTICIPANT] = (
    metrics[MetricsCols.PARTICIPANT]
    .astype(str).str.strip().str.lower()
)

# Keep only rows with both TOI and AOI populated (drop aggregate rows)
before = len(metrics)
metrics = metrics[
    metrics[MetricsCols.TOI].notna() &
    metrics[MetricsCols.AOI].notna()
].copy()
print(f"Rows after filtering to TOI+AOI rows: {len(metrics)} (removed {before - len(metrics)})")

# Replace Tobii sentinel dashes with NaN
metrics.replace('-', np.nan, inplace=True)
metrics.replace('- ', np.nan, inplace=True)

# Cast numeric columns
numeric_cols = metrics.columns.difference([
    MetricsCols.RECORDING, MetricsCols.PARTICIPANT, MetricsCols.TIMELINE,
    MetricsCols.TOI, MetricsCols.INTERVAL, MetricsCols.MEDIA, MetricsCols.AOI,
    MetricsCols.LAST_KEY_PRESS, MetricsCols.LAST_AOI_VIEWED, MetricsCols.AOI_AT_INTERVAL_END
])
for col in numeric_cols:
    metrics[col] = pd.to_numeric(metrics[col], errors='coerce')

# Confirm tasks present in Media column
print("\nMedia values found:")
print(metrics[MetricsCols.MEDIA].value_counts())

print("\nParticipants found:")
print(metrics[MetricsCols.PARTICIPANT].value_counts())

## 3. Load Data Export TSV

In [ ]:
# This file is ~31 MB — loading may take a moment
export_raw = pd.read_csv(DATAEXPORT_TSV, sep='\t', low_memory=False)
print(f"Shape: {export_raw.shape}")
export_raw.head(3)

## 4. Clean Data Export TSV

In [ ]:
gaze = export_raw.copy()

# Standardize participant ID
gaze[ExportCols.PARTICIPANT_NAME] = (
    gaze[ExportCols.PARTICIPANT_NAME]
    .astype(str).str.strip().str.lower()
)

# Replace Tobii string sentinels
sentinel_cols = [
    ExportCols.GAZE_X, ExportCols.GAZE_Y,
    ExportCols.FIXATION_X, ExportCols.FIXATION_Y,
    ExportCols.PUPIL_LEFT, ExportCols.PUPIL_RIGHT, ExportCols.PUPIL_FILTERED,
    ExportCols.EYE_OPEN_LEFT, ExportCols.EYE_OPEN_RIGHT, ExportCols.EYE_OPEN_FILT,
]
gaze[sentinel_cols] = gaze[sentinel_cols].replace('-', np.nan).replace('- ', np.nan)
for col in sentinel_cols:
    gaze[col] = pd.to_numeric(gaze[col], errors='coerce')

# Cast eye movement duration
gaze[ExportCols.EYE_MOVEMENT_DURATION] = pd.to_numeric(
    gaze[ExportCols.EYE_MOVEMENT_DURATION], errors='coerce'
)

# Validate Eye movement type
invalid_types = gaze[ExportCols.EYE_MOVEMENT_TYPE].dropna()
invalid_types = invalid_types[~invalid_types.isin(VALID_EYE_MOVEMENT_TYPES)]
print(f"Unexpected Eye movement type values: {invalid_types.unique()[:10]}")

# Cast AOI hit columns to int8
aoi_hit_cols = [c for c in gaze.columns if c.startswith(ExportCols.AOI_HIT_PREFIX)]
print(f"\nAOI hit columns found: {len(aoi_hit_cols)}")
gaze[aoi_hit_cols] = gaze[aoi_hit_cols].replace('-', 0).fillna(0).astype('int8')

## 5. Data Quality: Per-Participant Valid Gaze Rate

In [ ]:
# Valid = row has a valid Gaze point X (not NaN)
gaze['_valid_sample'] = gaze[ExportCols.GAZE_X].notna().astype(int)

quality = (
    gaze.groupby(ExportCols.PARTICIPANT_NAME)['_valid_sample']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'valid_samples', 'count': 'total_samples'})
)
quality['valid_rate'] = quality['valid_samples'] / quality['total_samples']
quality = quality.sort_values('valid_rate')

flagged = quality[quality['valid_rate'] < 0.70]
print(f"Participants below 70% valid gaze: {len(flagged)}")
display(quality)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
quality['valid_rate'].sort_values().plot(kind='bar', ax=ax, color='steelblue')
ax.axhline(0.70, color='red', linestyle='--', label='70% threshold')
ax.set_title('Per-Participant Valid Gaze Sample Rate')
ax.set_ylabel('Valid rate')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/01_gaze_quality.png', dpi=150)
plt.show()

## 6. Participant ID Alignment Check

In [ ]:
metrics_participants = set(metrics[MetricsCols.PARTICIPANT].unique())
gaze_participants    = set(gaze[ExportCols.PARTICIPANT_NAME].unique())

only_in_metrics = metrics_participants - gaze_participants
only_in_gaze    = gaze_participants - metrics_participants

print(f"Participants in Metrics only: {only_in_metrics}")
print(f"Participants in Data export only: {only_in_gaze}")
print(f"Shared participants: {len(metrics_participants & gaze_participants)}")

## 7. Save to Parquet

In [ ]:
metrics.drop(columns=['_valid_sample'], errors='ignore').to_parquet(METRICS_CLEAN_PKL, index=False)
gaze.drop(columns=['_valid_sample'], errors='ignore').to_parquet(RAWGAZE_CLEAN_PKL, index=False)

print(f"Saved: {METRICS_CLEAN_PKL}")
print(f"  Shape: {metrics.shape}")
print(f"Saved: {RAWGAZE_CLEAN_PKL}")
print(f"  Shape: {gaze.shape}")